In [6]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.options import prepare_options, build_forwards_options_comparison
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, prepare_pillars

from okx.features import add_tenor, parse_option
from okx.recipes.helpers import early_roll

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite",
    batch_days=5
)

dates = [date(2025, 9, 1)]

In [9]:
early_roll_fn = early_roll(min_time_to_expiry_hours=2.0)

start_time = datetime.now()

options_df = store.get(
    inst_family='BTC-USD',
    inst_type='OPTION',
    dates=dates,
    depth=1,
    features=['trim', 'strip']
)

time_1 = datetime.now()
print(f"Time taken to fetch options: {time_1 - start_time}")

options_df = add_tenor(options_df, inst_type='OPTION')
time_2 = datetime.now()
print(f"Time taken to add tenor: {time_2 - time_1}")

options_df = early_roll_fn(options_df)
time_3 = datetime.now()
print(f"Time taken to apply early roll: {time_3 - time_2}")

options_df = parse_option(options_df)
time_4 = datetime.now()
print(f"Time taken to parse options: {time_4 - time_3}")

options_df = options_df.collect()

time_5 = datetime.now()
print(f"Time taken to collect options: {time_5 - time_4}")

Time taken to fetch options: 0:00:00.003416
Time taken to add tenor: 0:00:00.000255
Time taken to apply early roll: 0:00:00.000067
Time taken to parse options: 0:00:00.000067
Time taken to collect options: 0:00:06.131955


In [10]:
options_df.head()

timeMs,symbol,bid_1_px,ask_1_px,expiry,T,strike,opt_type
i64,str,f64,f64,i64,f64,i64,str
1756685129286,"""BTC-USD-250902-100000-C.OK""",0.005,null,1756800000000,0.003643,100000,"""C"""
1756685150474,"""BTC-USD-250902-100000-C.OK""",0.005,null,1756800000000,0.003642,100000,"""C"""
1756685190095,"""BTC-USD-250902-100000-C.OK""",0.005,null,1756800000000,0.003641,100000,"""C"""
1756685207083,"""BTC-USD-250902-100000-C.OK""",0.005,null,1756800000000,0.00364,100000,"""C"""
1756686365156,"""BTC-USD-250902-100000-C.OK""",0.005,null,1756800000000,0.003603,100000,"""C"""


In [ ]:
print(options_df.shape)
print(options_df.head())
options_df_filtered = options_df.filter((pl.col('bid_1_px').is_not_null()) & (pl.col('ask_1_px').is_not_null()))
print(options_df_filtered.shape)
print(options_df_filtered.head())
duplicate_counts = (
    options_df_filtered
    .group_by([col for col in options_df_filtered.columns])
    .count()
    .filter(pl.col("count") > 1)
    .sort("count", descending=True)
)

print("Duplicate rows and their counts:")
print(duplicate_counts)


(56712547, 4)
shape: (5, 4)
┌───────────────┬────────────────────────────┬──────────┬──────────┐
│ timeMs        ┆ symbol                     ┆ bid_1_px ┆ ask_1_px │
│ ---           ┆ ---                        ┆ ---      ┆ ---      │
│ i64           ┆ str                        ┆ f64      ┆ f64      │
╞═══════════════╪════════════════════════════╪══════════╪══════════╡
│ 1756685129286 ┆ BTC-USD-250902-100000-C.OK ┆ 0.005    ┆ null     │
│ 1756685150474 ┆ BTC-USD-250902-100000-C.OK ┆ 0.005    ┆ null     │
│ 1756685190095 ┆ BTC-USD-250902-100000-C.OK ┆ 0.005    ┆ null     │
│ 1756685207083 ┆ BTC-USD-250902-100000-C.OK ┆ 0.005    ┆ null     │
│ 1756686365156 ┆ BTC-USD-250902-100000-C.OK ┆ 0.005    ┆ null     │
└───────────────┴────────────────────────────┴──────────┴──────────┘
(56331743, 4)
shape: (5, 4)
┌───────────────┬────────────────────────────┬──────────┬──────────┐
│ timeMs        ┆ symbol                     ┆ bid_1_px ┆ ask_1_px │
│ ---           ┆ ---                        ┆ 

/var/folders/fl/fdwpgpsx7fx15p__07t93bhr0000gn/T/ipykernel_12483/3014190339.py:9: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


Duplicate rows and their counts:
shape: (4_145, 5)
┌───────────────┬────────────────────────────┬──────────┬──────────┬───────┐
│ timeMs        ┆ symbol                     ┆ bid_1_px ┆ ask_1_px ┆ count │
│ ---           ┆ ---                        ┆ ---      ┆ ---      ┆ ---   │
│ i64           ┆ str                        ┆ f64      ┆ f64      ┆ u32   │
╞═══════════════╪════════════════════════════╪══════════╪══════════╪═══════╡
│ 1756697574577 ┆ BTC-USD-250905-106000-P.OK ┆ 0.01     ┆ 0.0105   ┆ 5     │
│ 1756697574555 ┆ BTC-USD-250903-110000-C.OK ┆ 0.0032   ┆ 0.0033   ┆ 5     │
│ 1756741949409 ┆ BTC-USD-250902-108000-P.OK ┆ 0.0027   ┆ 0.0028   ┆ 4     │
│ 1756723153646 ┆ BTC-USD-250903-108000-P.OK ┆ 0.0085   ┆ 0.009    ┆ 4     │
│ 1756689466848 ┆ BTC-USD-250905-104000-P.OK ┆ 0.0048   ┆ 0.0049   ┆ 4     │
│ …             ┆ …                          ┆ …        ┆ …        ┆ …     │
│ 1756812245955 ┆ BTC-USD-250905-108000-P.OK ┆ 0.0055   ┆ 0.006    ┆ 2     │
│ 1756746725170 ┆ BTC-USD